# Chapter 3 — Time Zones and Daylight Saving Time
**Working with Dates and Times in Python**

Topics covered:
- What UTC is and why it matters
- UTC offsets and timezone-aware datetimes
- The `tz` database (`dateutil`) — the professional approach
- `astimezone()` vs `replace()` for timezone handling
- Daylight Saving Time (DST) — spring forward, fall back
- Ambiguous times and `tz.enfold()`

---
> **Why this matters for data analysts / data scientists:**  
> Real-world datasets almost always contain timestamps from multiple time zones (server logs, IoT sensors, financial transactions, user activity). Getting timezone handling wrong leads to incorrect aggregations, wrong time-series plots, and bad business decisions.

---
## 1. What is UTC and why does it matter?

**UTC (Coordinated Universal Time)** is the universal reference point for all time zones:
- West of the prime meridian → UTC **minus** hours (e.g. New York = UTC-5)
- East of the prime meridian → UTC **plus** hours (e.g. India = UTC+5:30)

**Best practice for data work:** Always store timestamps in UTC internally, convert to local time only for display. This avoids bugs when data spans multiple time zones.

### Naive vs Aware datetimes
| Type | Has timezone? | Risk |
|------|--------------|------|
| Naive | No | Dangerous — Python doesn't know what timezone it is |
| Aware | Yes (tzinfo) | Safe — unambiguous point in time |

---
## 2. UTC offsets — the manual approach

Use `timezone(timedelta(hours=N))` to define a fixed UTC offset.  
Pass it to a datetime via the `tzinfo` argument.

In [1]:
from datetime import datetime, timedelta, timezone

# Define timezone offsets manually
ET  = timezone(timedelta(hours=-5))    # US Eastern Standard Time
IST = timezone(timedelta(hours=5, minutes=30))  # India Standard Time
UTC = timezone.utc                     # UTC built-in constant

# Create a timezone-AWARE datetime (notice tzinfo parameter)
dt = datetime(2017, 12, 30, 15, 9, 3, tzinfo=ET)
print(dt)          # 2017-12-30 15:09:03-05:00
                   # The -05:00 at the end shows the UTC offset

2017-12-30 15:09:03-05:00


---
## 3. Converting between time zones — `astimezone()` vs `replace()`

These two methods look similar but do very different things:

| Method | What it does | Use when... |
|--------|-------------|-------------|
| `.astimezone(tz)` | Converts to new tz, **adjusts the clock** | You want the same moment in a different zone |
| `.replace(tzinfo=tz)` | Just changes the label, **keeps the clock as-is** | You know the stored time was wrong/naive |

In [2]:
from datetime import datetime, timedelta, timezone

ET  = timezone(timedelta(hours=-5))
IST = timezone(timedelta(hours=5, minutes=30))

dt = datetime(2017, 12, 30, 15, 9, 3, tzinfo=ET)
print("Original (ET):        ", dt)   # 2017-12-30 15:09:03-05:00

# astimezone → same moment, clock adjusted → time changes
print("astimezone to IST:    ", dt.astimezone(IST))   # 2017-12-31 01:39:03+05:30
print("astimezone to UTC:    ", dt.astimezone(timezone.utc))  # 2017-12-30 20:09:03+00:00

# replace → just slaps a new label on, clock stays same → DANGEROUS!
print("replace tzinfo to UTC:", dt.replace(tzinfo=timezone.utc))  # 2017-12-30 15:09:03+00:00
# ↑ This says 15:09 UTC, which is a DIFFERENT moment in time from 15:09 ET!

print()
print("Rule: use astimezone() to CONVERT, use replace() only to FIX a naive datetime")

Original (ET):         2017-12-30 15:09:03-05:00
astimezone to IST:     2017-12-31 01:39:03+05:30
astimezone to UTC:     2017-12-30 20:09:03+00:00
replace tzinfo to UTC: 2017-12-30 15:09:03+00:00

Rule: use astimezone() to CONVERT, use replace() only to FIX a naive datetime


---
## 4. The `tz` database — the professional approach

Manual UTC offsets (`timezone(timedelta(hours=-5))`) have a big problem: **they don't handle Daylight Saving Time automatically**.  

The solution is the **tz database** (also called the Olson/IANA database) — a globally maintained list of every timezone's full history including DST changes.  
Access it in Python via `dateutil.tz`.

**Format:** `'Continent/City'`  
Examples: `'America/New_York'`, `'Europe/London'`, `'Asia/Kolkata'`, `'Africa/Accra'`

In [3]:
# pip install python-dateutil  (usually already installed)
from datetime import datetime
from dateutil import tz

# Get a timezone object from the tz database
eastern = tz.gettz('America/New_York')
london  = tz.gettz('Europe/London')
india   = tz.gettz('Asia/Kolkata')

# Create timezone-aware datetimes
last  = datetime(2017, 12, 30, 15, 9, 3,  tzinfo=eastern)  # Winter → EST (-05:00)
first = datetime(2017, 10,  1, 15, 23, 25, tzinfo=eastern)  # Autumn → EDT (-04:00)

print(last)    # 2017-12-30 15:09:03-05:00  ← -5 (EST, no DST)
print(first)   # 2017-10-01 15:23:25-04:00  ← -4 (EDT, during DST)

# Notice: SAME timezone object, but Python automatically used -5 vs -4
# because dateutil knows when DST was active!

2017-12-30 15:09:03-05:00
2017-10-01 15:23:25-04:00


---
## 5. Daylight Saving Time — Spring Forward

In the US, clocks **spring forward** 1 hour at 2:00 AM on the second Sunday of March:  
- 1:59:59 AM EST → next second becomes 3:00:00 AM EDT  
- The hour from 2:00–2:59 AM **does not exist** that day!

**Without timezone awareness**, Python gets the duration wrong:

In [ ]:
from datetime import datetime, timezone, timedelta
from dateutil import tz

# --- NAIVE (wrong!) ---
spring_naive_159am = datetime(2017, 3, 12, 1, 59, 59)
spring_naive_3am   = datetime(2017, 3, 12, 3,  0,  0)

naive_diff = (spring_naive_3am - spring_naive_159am).total_seconds()
print(f"Naive difference: {naive_diff} seconds")   # 3601.0 — WRONG! Clock jumped 1 hour
# Python thinks 61 minutes passed, but only 1 second actually passed

print()

# --- TIMEZONE-AWARE with tz database (correct!) ---
eastern = tz.gettz('America/New_York')

spring_aware_159am = datetime(2017, 3, 12, 1, 59, 59, tzinfo=eastern)  # EST
spring_aware_3am   = datetime(2017, 3, 12, 3,  0,  0, tzinfo=eastern)  # EDT

aware_diff = (spring_aware_3am - spring_aware_159am).total_seconds()
print(f"Aware difference:  {aware_diff} second")   # 1.0 — CORRECT!
# dateutil knows 1:59:59 EST and 3:00:00 EDT are only 1 second apart

---
## 6. Daylight Saving Time — Fall Back (ambiguous times)

In autumn, clocks **fall back** 1 hour at 2:00 AM:  
- 1:59:59 AM EDT → next second becomes 1:00:00 AM EST  
- The hour from 1:00–1:59 AM **happens TWICE** that day!

This creates **ambiguous timestamps** — if you see `2017-11-05 01:30:00`, is that EDT or EST?  
Use `tz.datetime_ambiguous()` to detect this, and `tz.enfold()` to mark the second occurrence.

In [ ]:
from datetime import datetime
from dateutil import tz

eastern = tz.gettz('US/Eastern')

# Both of these look identical as naive datetimes
first_1am  = datetime(2017, 11, 5, 1, 0, 0, tzinfo=eastern)  # EDT (before fall-back)
second_1am = datetime(2017, 11, 5, 1, 0, 0, tzinfo=eastern)  # EST (after fall-back)

# Check if this time is ambiguous
print("Is ambiguous?", tz.datetime_ambiguous(first_1am))   # True

# Mark the second occurrence with enfold()
second_1am = tz.enfold(second_1am)

# Before converting to UTC they look the same (0 seconds apart)
print("Before UTC conversion:", (first_1am - second_1am).total_seconds())  # 0.0

# Convert both to UTC — now the real 1-hour gap is revealed
first_1am_utc  = first_1am.astimezone(tz.UTC)
second_1am_utc = second_1am.astimezone(tz.UTC)

print("After UTC conversion:", (second_1am_utc - first_1am_utc).total_seconds())  # 3600.0
print()
print("first_1am  in UTC:", first_1am_utc)   # 05:00 UTC (EDT = UTC-4)
print("second_1am in UTC:", second_1am_utc)  # 06:00 UTC (EST = UTC-5)

---
## 7. Real-world data analyst workflow

A typical workflow when working with timestamps from multiple time zones:

In [4]:
from datetime import datetime
from dateutil import tz

# Step 1: Raw timestamps arrive as strings (e.g. from a CSV / database)
raw_timestamps = [
    "2017-10-01 15:23:25",   # US user
    "2017-10-01 20:23:25",   # UK user
    "2017-10-02 01:53:25",   # India user
]

# Step 2: Parse strings to datetime
parsed = [datetime.strptime(ts, "%Y-%m-%d %H:%M:%S") for ts in raw_timestamps]

# Step 3: Localize — tell Python what timezone each timestamp is in
timezones = [
    tz.gettz('America/New_York'),
    tz.gettz('Europe/London'),
    tz.gettz('Asia/Kolkata'),
]
localized = [dt.replace(tzinfo=tz_) for dt, tz_ in zip(parsed, timezones)]
# Note: .replace() is correct here — we're LABELING naive datetimes,
# not converting (we know what timezone the source data was in)

# Step 4: Convert everything to UTC for consistent comparison/storage
utc_times = [dt.astimezone(tz.UTC) for dt in localized]

print("Localized times:")
for dt in localized:
    print(" ", dt)

print("\nAll converted to UTC (same moment in time):")
for dt in utc_times:
    print(" ", dt)
# All three should now show the same UTC time → they happened simultaneously!

Localized times:
  2017-10-01 15:23:25-04:00
  2017-10-01 20:23:25+01:00
  2017-10-02 01:53:25+05:30

All converted to UTC (same moment in time):
  2017-10-01 19:23:25+00:00
  2017-10-01 19:23:25+00:00
  2017-10-01 20:23:25+00:00


---
## 8. Common timezone names reference

| Region | tz database name |
|--------|------------------|
| New York / Eastern US | `'America/New_York'` |
| Chicago / Central US | `'America/Chicago'` |
| Denver / Mountain US | `'America/Denver'` |
| Los Angeles / Pacific US | `'America/Los_Angeles'` |
| London (UK) | `'Europe/London'` |
| Paris / Berlin | `'Europe/Paris'` / `'Europe/Berlin'` |
| Mumbai / India | `'Asia/Kolkata'` |
| Singapore | `'Asia/Singapore'` |
| Tokyo | `'Asia/Tokyo'` |
| Sydney | `'Australia/Sydney'` |
| UTC | `'UTC'` or `tz.UTC` |

In [ ]:
from datetime import datetime
from dateutil import tz

# The same moment in time shown in different timezones
utc_now = datetime(2017, 12, 30, 20, 0, 0, tzinfo=tz.UTC)

zones = [
    ('UTC',                 tz.UTC),
    ('New York',            tz.gettz('America/New_York')),
    ('Los Angeles',         tz.gettz('America/Los_Angeles')),
    ('London',              tz.gettz('Europe/London')),
    ('India',               tz.gettz('Asia/Kolkata')),
    ('Tokyo',               tz.gettz('Asia/Tokyo')),
]

print(f"{'City':<15} {'Local time':<30} {'UTC offset'}")
print("-" * 60)
for name, zone in zones:
    local = utc_now.astimezone(zone)
    offset = local.strftime('%z')
    print(f"{name:<15} {local.strftime('%Y-%m-%d %H:%M:%S'):<30} UTC{offset}")

---
## Quick Reference Summary

| Task | Code |
|------|------|
| Define a UTC offset | `timezone(timedelta(hours=-5))` |
| Get tz from database | `tz.gettz('America/New_York')` |
| Make a datetime timezone-aware | `datetime(..., tzinfo=eastern)` |
| Convert to another timezone | `dt.astimezone(new_tz)` |
| Label a naive datetime (fix it) | `dt.replace(tzinfo=eastern)` |
| Convert to UTC | `dt.astimezone(tz.UTC)` |
| Check if time is ambiguous (DST) | `tz.datetime_ambiguous(dt)` |
| Mark second occurrence of ambiguous time | `tz.enfold(dt)` |

### Golden rules for data work
1. Always use `tz.gettz()` over manual `timezone(timedelta(...))` — it handles DST automatically
2. Store all timestamps in UTC; convert to local only for display
3. Use `astimezone()` to convert, `replace()` only to label naive datetimes
4. Watch out for ambiguous times during DST fall-back — use `tz.enfold()` to disambiguate

In [ ]:
# Practice cell — try your own examples here!
from datetime import datetime
from dateutil import tz

# Your code below:
